# Comparação de Experimentos: MOEA + DVL vs Apenas MOEA

In [1]:
import numpy as np
import random
import time
import pandas as pd
import os

# Importando os Problemas, Algoritmos e Indicador de Qualidade
from src.problems.DTLZ import DTLZ1, DTLZ2, DTLZ3, DTLZ4
from src.MOEAs.MOEAD import MOEAD
from src.MOEAs.NSGAII import NSGAII
from src.MOEAs.NSGAIII import NSGAIII

from src.dvl.DVL import DVLFramework
from src.dvl.models.Linear import LinearModel
from src.QualityIndicator import HV, IGD

from src.MOEAs.mutations.PolynomialMutation import PolynomialMutation
from src.MOEAs.crossovers.SBXCrossover import SBXCrossover
from src.BinaryTournament import BinaryTournament
from src.MOEAs.sparsities.CrowdingDistance import CrowdingDistance

# Pontos de referência para o cálculo do Hypervolume
HV_REFERENCE_POINTS = {
    ("DTLZ1", 3): np.array([1.0, 1.0, 1.0], dtype=float),
    ("DTLZ1", 10): np.array([5.0] * 10, dtype=float),
    ("DTLZ2", 3): np.array([2.0, 2.0, 2.0], dtype=float),
    ("DTLZ2", 10): np.array([2.0] * 10, dtype=float),
    ("DTLZ3", 3): np.array([2.0, 2.0, 2.0], dtype=float),
    ("DTLZ3", 10): np.array([2.0] * 10, dtype=float),
    ("DTLZ4", 3): np.array([2.0, 2.0, 2.0], dtype=float),
    ("DTLZ4", 10): np.array([2.0] * 10, dtype=float),
}

In [2]:
def run_with_dvl(
    problem_class,      # Classe do problema
    moea_class,         # Classe do algoritmo evolucionário
    model_class,        # Classe do modelo de aprendizado de máquina
    m,                  # Número de objetivos do problema (M)
    k,                  # Parâmetro k do problema DTLZ
    max_evaluations,    # Número máximo de avaliações
    sample_size,        # Tamanho da amostra inicial (LHS) para treinar o DVL
    seed=42,
    run_moea=True
):
    np.random.seed(seed)
    random.seed(seed)
    
    problem = problem_class(numberOfObjectives=m, k=k)
    model = model_class()
    
    div_dict = {2: 99, 3: 12, 10: 3}
    num_div = div_dict.get(m, 12)
    
    framework = DVLFramework(
        pop_size=sample_size,
        max_eval=max_evaluations,
        ClassMoea=moea_class,
        model=model,
        problem=problem,
        reference_point_divisions=num_div,
        sampling_seed=seed,
        objective_transform="direction",
        run_moea=run_moea
    )
    
    population = framework.execute()
    return population

In [3]:
def run_without_dvl(
    problem_class,  # Classe do problema (ex: DTLZ1, DTLZ2)
    moea_class,     # Classe do algoritmo evolucionário (ex: NSGAIII, NSGAII)
    m,              # Número de objetivos do problema (M)
    k,              # Parâmetro k do problema DTLZ
    max_evaluations,# Número máximo de avaliações
    seed=42
):
    import inspect
    from src.Util import ReferencePoint

    np.random.seed(seed)
    random.seed(seed)

    problem = problem_class(numberOfObjectives=m, k=k)
    
    div_dict = {2: 99, 3: 12, 10: 3}
    num_div = div_dict.get(m, 12)
    
    crossover = SBXCrossover(20.0, 0.9)
    mutation_probability = 1.0 / problem.numberOfDecisionVariables
    mutation = PolynomialMutation(mutation_probability, 20.0)
    selection = BinaryTournament()
    sparsity = CrowdingDistance()
    
    # Gera pontos de referência para deduzir tamanho de população se o algoritmo precisar
    ref_points = ReferencePoint().generateReferencePoints(m, num_div)
    pop_size = len(ref_points)

    # Identificar assinatura do construtor dinamicamente
    sig = inspect.signature(moea_class)
    params = sig.parameters
    
    kwargs = {}
    if "problem" in params:
        kwargs["problem"] = problem
    if "maxEvaluations" in params:
        kwargs["maxEvaluations"] = max_evaluations
    if "crossover" in params:
        kwargs["crossover"] = crossover
    if "mutation" in params:
        kwargs["mutation"] = mutation
    if "selection" in params:
        kwargs["selection"] = selection
    if "sparsity" in params:
        kwargs["sparsity"] = sparsity
    if "numberOfDivisions" in params:
        kwargs["numberOfDivisions"] = num_div
    if "populationSize" in params:
        kwargs["populationSize"] = pop_size
    if "offSpringPopulationSize" in params:
        kwargs["offSpringPopulationSize"] = pop_size if pop_size % 2 == 0 else pop_size + 1
        
    moea = moea_class(**kwargs)
    population = moea.execute()
    if population is None:
        population = moea.population
    
    return population

In [4]:
def build_experiment_configs():
    configs = []
    
    # Parâmetros para 3 objetivos (k=10)
    m3_configs = {
        "DTLZ1": {"e": [250, 500, 1000, 1500, 10000], "s": [159, 227, 250, 300, 300]},
        "DTLZ2": {"e": [250, 500, 1000, 1500, 10000], "s": [159, 227, 600, 600, 600]},
        "DTLZ3": {"e": [250, 500, 1000, 1500, 10000], "s": [159, 227, 300, 300, 300]},
        "DTLZ4": {"e": [250, 500, 1000, 1500, 10000], "s": [159, 410, 410, 410, 500]},
        
    }
    
    # Parâmetros para 10 objetivos (k=1)
    m10_configs = {
        "DTLZ1": {"e": [250, 500, 1000, 1500, 10000], "s": [50, 112, 200, 200, 300]},
        "DTLZ2": {"e": [250, 500, 1000, 1500, 10000], "s": [50, 280, 300, 300, 300]},
        "DTLZ3": {"e": [250, 500, 1000, 1500, 10000], "s": [50, 112, 200, 200, 300]},
        "DTLZ4": {"e": [250, 500, 1000, 1500, 10000], "s": [50, 280, 300, 300, 300]},
    }
    
    # Gerando para m = 3
    for problem_name, data in m3_configs.items():
        for e, s in zip(data["e"], data["s"]):
            configs.append({
                "problem_name": problem_name,
                "m": 3,
                "problem_k": 10,
                "max_evaluations": e,
                "sample_size": s,
                "label": f"{problem_name}_m3_e{e}_s{s}"
            })
            
    # Gerando para m = 10
    for problem_name, data in m10_configs.items():
        for e, s in zip(data["e"], data["s"]):
            configs.append({
                "problem_name": problem_name,
                "m": 10,
                "problem_k": 1,
                "max_evaluations": e,
                "sample_size": s,
                "label": f"{problem_name}_m10_e{e}_s{s}"
            })
            
    return configs

In [5]:
def run_experiment(config, problem_class, algo_class, mode, seed):
    """Executa um único experimento, lidando com erros, persistindo dados em CSV e retornando as métricas."""
    import os
    label = config["label"]
    problem_name = config["problem_name"]
    m = config["m"]
    k = config["problem_k"]
    max_evals = config["max_evaluations"]
    sample_size = config["sample_size"]
    
    result = {
        "label": label,
        "problem": problem_name,
        "algorithm": algo_class.__name__,
        "mode": mode,
        "m": m,
        "problem_k": k,
        "max_evaluations": max_evals,
        "sample_size": sample_size,
        "seed": seed,
        "cpu_time_seconds": 0.0,
        "population_size_final": None,
        "hypervolume": 0.0,
        "igd": 0.0,
        "status": "success",
        "error_message": ""
    }
    
    population = None
    start_cpu = time.process_time()
    try:
        if mode == "pure_moea":
            population = run_without_dvl(
                problem_class=problem_class,
                moea_class=algo_class,
                m=m,
                k=k,
                max_evaluations=max_evals,
                seed=seed
            )
        elif mode == "dvl_framework":
            population = run_with_dvl(
                problem_class=problem_class,
                moea_class=algo_class,
                model_class=LinearModel,
                m=m,
                k=k,
                max_evaluations=max_evals,
                sample_size=sample_size,
                seed=seed,
                run_moea=True
            )
        else:
            raise ValueError(f"Mode desconhecido: {mode}")
            
        end_cpu = time.process_time()
        result["cpu_time_seconds"] = end_cpu - start_cpu
        result["population_size_final"] = len(population)
        
        # Cálculo do Hypervolume Normalizado
        ref_point = HV_REFERENCE_POINTS.get((problem_name, m))
        if ref_point is not None:
            objectives_list = [sol.objectives for sol in population] # type: ignore
            indicator = HV(referencePoint=ref_point)
            hv_val = indicator.calculate(objectives_list)
            result["hypervolume"] = hv_val / np.prod(ref_point)
            
        # Cálculo do IGD Normalizado
        ref_front_path = f"resources/ReferenceFronts/DTLZ/{problem_name}.{m}D.csv"
        if os.path.exists(ref_front_path):
            ref_front = np.loadtxt(ref_front_path, delimiter=",")
            objectives_list = np.array([sol.objectives for sol in population]) # type: ignore
            
            # Normalização (divisão por 0.5 para DTLZ1, por 1.0 para DTLZ2-4)
            norm_factor = 0.5 if problem_name == "DTLZ1" else 1.0
            front_normalized = objectives_list / norm_factor
            ref_front_normalized = ref_front / norm_factor
            
            indicator_igd = IGD(ref_front_normalized.tolist())
            result["igd"] = indicator_igd.calculate(front_normalized.tolist())
            
    except Exception as e:
        end_cpu = time.process_time()
        result["cpu_time_seconds"] = end_cpu - start_cpu
        result["status"] = "error"
        result["error_message"] = str(e)
        result["hypervolume"] = np.nan
        result["igd"] = np.nan
        
    # Criar pasta out se não existir
    os.makedirs("out", exist_ok=True)
    
    # 1. Persistir métricas gerais (incremental append)
    df_metrics = pd.DataFrame([result])
    metrics_csv_path = "out/all_results.csv"
    df_metrics.to_csv(metrics_csv_path, mode='a', header=not os.path.exists(metrics_csv_path), index=False)
    
    # 2. Persistir os dados da população final (objetivos e variáveis de decisão)
    if result["status"] == "success" and population is not None:
        pop_data = []
        for sol in population:
            row = {}
            for idx, obj in enumerate(sol.objectives): # type: ignore
                row[f"obj_{idx}"] = obj
            for idx, var in enumerate(sol.decisionVariables): # type: ignore
                row[f"var_{idx}"] = var
            pop_data.append(row)
        pop_df = pd.DataFrame(pop_data)
        pop_csv_path = f"out/{label}_{algo_class.__name__}_{mode}_seed_{seed}_population.csv"
        pop_df.to_csv(pop_csv_path, index=False)
        
    return result

In [6]:
# Configurando o loop principal de experimentos
configs = build_experiment_configs()

problem_classes = {
    "DTLZ1": DTLZ1,
    "DTLZ2": DTLZ2,
    "DTLZ3": DTLZ3,
    "DTLZ4": DTLZ4
}
algorithms = [MOEAD, NSGAII, NSGAIII]

# Configurações de Execução
SEED = 42
NUM_RUNS = 20
results = []

# Carrega execuções anteriores para evitar reexecutá-las e preencher o results
existing_runs = set()
csv_path = "out/all_results.csv"
if os.path.exists(csv_path):
    try:
        df_existing = pd.read_csv(csv_path)
        results = df_existing.to_dict(orient="records")
        for r in results:
            # Chave única para identificar se já foi executado
            key = (str(r["label"]), str(r["algorithm"]), str(r["mode"]), int(r["seed"]))
            existing_runs.add(key)
        print(f"Carregados {len(results)} registros anteriores de {csv_path}.")
    except Exception as e:
        print(f"Erro ao carregar execuções anteriores: {e}")

# Calcular o total de execuções ativas
active_configs = [c for c in configs if c["problem_name"] in problem_classes]
total_runs = len(active_configs) * len(algorithms) * NUM_RUNS * 2
run_counter = 0

print(f"Total de execuções a realizar: {total_runs}\n")

for config in configs:
    problem_class = problem_classes.get(config["problem_name"])
    if not problem_class:
        continue
        
    for algo_class in algorithms:
        for i in range(NUM_RUNS):
            # 1. MOEA Puro
            run_counter += 1
            key_pure = (str(config["label"]), str(algo_class.__name__), "pure_moea", int(SEED + i))
            if key_pure in existing_runs:
                pass
            else:
                print(f"[{run_counter}/{total_runs}] Executando {config['label']} | {algo_class.__name__} | pure_moea... (Faltam {total_runs - run_counter})")
                res = run_experiment(config, problem_class, algo_class, mode="pure_moea", seed=SEED + i)
                results.append(res)
            
            # 2. DVL + MOEA
            run_counter += 1
            key_dvl = (str(config["label"]), str(algo_class.__name__), "dvl_framework", int(SEED + i))
            if key_dvl in existing_runs:
                pass
            else:
                print(f"[{run_counter}/{total_runs}] Executando {config['label']} | {algo_class.__name__} | dvl_framework... (Faltam {total_runs - run_counter})")
                res_dvl = run_experiment(config, problem_class, algo_class, mode="dvl_framework", seed=SEED + i)
                results.append(res_dvl)


Carregados 3076 registros anteriores de out/all_results.csv.
Total de execuções a realizar: 4800

[2958/4800] Executando DTLZ1_m10_e10000_s300 | NSGAII | dvl_framework... (Faltam 1842)
[2959/4800] Executando DTLZ1_m10_e10000_s300 | NSGAII | pure_moea... (Faltam 1841)
[2960/4800] Executando DTLZ1_m10_e10000_s300 | NSGAII | dvl_framework... (Faltam 1840)
[2961/4800] Executando DTLZ1_m10_e10000_s300 | NSGAIII | pure_moea... (Faltam 1839)
[2962/4800] Executando DTLZ1_m10_e10000_s300 | NSGAIII | dvl_framework... (Faltam 1838)
Evaluations: 0 de 9480...
[2963/4800] Executando DTLZ1_m10_e10000_s300 | NSGAIII | pure_moea... (Faltam 1837)
[2964/4800] Executando DTLZ1_m10_e10000_s300 | NSGAIII | dvl_framework... (Faltam 1836)
Evaluations: 0 de 9480...
[2965/4800] Executando DTLZ1_m10_e10000_s300 | NSGAIII | pure_moea... (Faltam 1835)
Evaluations: 2000 de 10000...
[2966/4800] Executando DTLZ1_m10_e10000_s300 | NSGAIII | dvl_framework... (Faltam 1834)
Evaluations: 0 de 9480...
Evaluations: 8000 de 

In [7]:
# Visualização dos Resultados das Execuções Raw
df = pd.DataFrame(results)

# Separação em dois dataframes: dentro do framework (dvl_framework) e fora do framework (pure_moea)
df_framework = df[df['mode'] == 'dvl_framework']
df_pure = df[df['mode'] == 'pure_moea']

print("Execuções Dentro do Framework (DVL + MOEA):")
display(df_framework)

print("\nExecuções Fora do Framework (Pure MOEA):")
display(df_pure)

Execuções Dentro do Framework (DVL + MOEA):


,label,problem,algorithm,mode,m,problem_k,max_evaluations,sample_size,seed,cpu_time_seconds,population_size_final,hypervolume,igd,status,error_message
1,DTLZ1_m3_e250_s159,DTLZ1,MOEAD,dvl_framework,3,10,250,159,42,0.031262,91.0,0.383151,0.632483,success,NaN
3,DTLZ1_m3_e250_s159,DTLZ1,MOEAD,dvl_framework,3,10,250,159,43,0.031060,91.0,0.539809,0.474607,success,NaN
5,DTLZ1_m3_e250_s159,DTLZ1,NSGAII,dvl_framework,3,10,250,159,42,0.032480,91.0,0.383151,0.632483,success,NaN
7,DTLZ1_m3_e250_s159,DTLZ1,NSGAII,dvl_framework,3,10,250,159,43,0.031263,91.0,0.539809,0.474607,success,NaN
9,DTLZ1_m3_e250_s159,DTLZ1,NSGAIII,dvl_framework,3,10,250,159,42,0.033284,91.0,0.383151,0.632483,success,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4910,DTLZ4_m10_e10000_s300,DTLZ4,NSGAIII,dvl_framework,10,1,10000,300,57,27.573756,187.0,0.999870,0.454177,success,
4912,DTLZ4_m10_e10000_s300,DTLZ4,NSGAIII,dvl_framework,10,1,10000,300,58,26.724497,203.0,0.999930,0.436978,success,
4914,DTLZ4_m10_e10000_s300,DTLZ4,NSGAIII,dvl_framework,10,1,10000,300,59,27.483883,209.0,0.999990,0.432010,success,
4916,DTLZ4_m10_e10000_s300,DTLZ4,NSGAIII,dvl_framework,10,1,10000,300,60,27.953251,202.0,0.999940,0.438883,success,



Execuções Fora do Framework (Pure MOEA):


,label,problem,algorithm,mode,m,problem_k,max_evaluations,sample_size,seed,cpu_time_seconds,population_size_final,hypervolume,igd,status,error_message
0,DTLZ1_m3_e250_s159,DTLZ1,MOEAD,pure_moea,3,10,250,159,42,0.045395,23.0,0.000000,174.258760,success,NaN
2,DTLZ1_m3_e250_s159,DTLZ1,MOEAD,pure_moea,3,10,250,159,43,0.042942,21.0,0.000000,199.110442,success,NaN
4,DTLZ1_m3_e250_s159,DTLZ1,NSGAII,pure_moea,3,10,250,159,42,0.060944,91.0,0.000000,279.927862,success,NaN
6,DTLZ1_m3_e250_s159,DTLZ1,NSGAII,pure_moea,3,10,250,159,43,0.057318,91.0,0.000000,299.834328,success,NaN
8,DTLZ1_m3_e250_s159,DTLZ1,NSGAIII,pure_moea,3,10,250,159,42,0.106262,88.0,0.162061,1.495410,success,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4909,DTLZ4_m10_e10000_s300,DTLZ4,NSGAIII,pure_moea,10,1,10000,300,57,28.388972,217.0,0.999950,0.439831,success,
4911,DTLZ4_m10_e10000_s300,DTLZ4,NSGAIII,pure_moea,10,1,10000,300,58,28.279637,201.0,0.999970,0.442904,success,
4913,DTLZ4_m10_e10000_s300,DTLZ4,NSGAIII,pure_moea,10,1,10000,300,59,27.437084,202.0,0.999970,0.429641,success,
4915,DTLZ4_m10_e10000_s300,DTLZ4,NSGAIII,pure_moea,10,1,10000,300,60,28.653062,207.0,0.999970,0.425011,success,


In [8]:
import pandas as pd
import os
import numpy as np
from src.QualityIndicator import IGD

pd.set_option("display.float_format", "{:.8f}".format)

# Cache de fronteiras de referência para não recalcular a cada linha
pareto_fronts_cache = {}

def get_cached_pareto_front(problem_name, m, k):
    key = (problem_name, m)
    if key in pareto_fronts_cache:
        return pareto_fronts_cache[key]
    
    ref_front = None
    ref_front_path = f"resources/ReferenceFronts/DTLZ/{problem_name}.{m}D.csv"
    if os.path.exists(ref_front_path):
        ref_front = np.loadtxt(ref_front_path, delimiter=",")
                
    pareto_fronts_cache[key] = ref_front
    return ref_front

def compute_row_igd(row):
    label = row["label"]
    algo = row["algorithm"]
    mode = row["mode"]
    seed = row["seed"]
    problem_name = row["problem"]
    m = row["m"]
    k = row["problem_k"]
    
    # Tentamos os dois padrões de nomenclatura de arquivos de população na pasta out
    paths = [
        f"out/{label}_{algo}_{mode}_seed_{seed}_population.csv",
        f"out/{label}_{mode}_seed_{seed}_population.csv"
    ]
    
    pop_path = None
    for p in paths:
        if os.path.exists(p):
            pop_path = p
            break
            
    if pop_path is None:
        return np.nan
        
    try:
        pop_df = pd.read_csv(pop_path)
        obj_cols = [c for c in pop_df.columns if c.startswith("obj_")]
        if not obj_cols:
            return np.nan
        front = pop_df[obj_cols].values
        
        ref_front = get_cached_pareto_front(problem_name, m, k)
        if ref_front is None:
            return np.nan
            
        # Normalização (divisão por 0.5 para DTLZ1, por 1.0 para DTLZ2-4)
        norm_factor = 0.5 if problem_name == "DTLZ1" else 1.0
        front_normalized = front / norm_factor
        ref_front_normalized = ref_front / norm_factor
        
        indicator_igd = IGD(ref_front_normalized.tolist())
        return indicator_igd.calculate(front_normalized.tolist())
    except Exception as e:
        return np.nan

csv_path = "out/all_results.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)

if not df.empty and "hypervolume" in df.columns:
    # O IGD já está calculado no all_results.csv e é idêntico ao recalculado.
    # Evitamos a recalculação lenta lendo milhares de arquivos.
    pass
    
    summary_df = df.groupby(["problem", "label", "algorithm", "m", "max_evaluations", "sample_size", "mode"]).agg(
        hv_mean=("hypervolume", "mean"),
        hv_std=("hypervolume", "std"),
        igd_mean=("igd", "mean"),
        igd_std=("igd", "std"),
        cpu_time_mean=("cpu_time_seconds", "mean"),
        cpu_time_std=("cpu_time_seconds", "std"),
        successful_runs=("status", lambda x: (x == "success").sum())
    ).reset_index()
    
    # Ordenando por problema, algoritmo, max_evaluations (crescente) e sample_size (crescente, desempate)
    summary_df = summary_df.sort_values(by=["problem", "algorithm", "max_evaluations", "sample_size"], ascending=True)
    
    # Separação do resumo estatístico em dois dataframes
    summary_framework = summary_df[summary_df['mode'] == 'dvl_framework']
    summary_pure = summary_df[summary_df['mode'] == 'pure_moea']
    
    print("Resumo Dentro do Framework (DVL + MOEA):")
    display(summary_framework)
    
    print("\nResumo Fora do Framework:")
    display(summary_pure)
else:
    print("Nenhum resultado disponível para resumir.")

Resumo Dentro do Framework (DVL + MOEA):


,problem,label,algorithm,m,max_evaluations,sample_size,mode,hv_mean,hv_std,igd_mean,igd_std,cpu_time_mean,cpu_time_std,successful_runs
18,DTLZ1,DTLZ1_m10_e250_s50,MOEAD,10,250,50,dvl_framework,0.81774296,0.24235187,2.66861938,3.01009233,0.10924594,0.00458918,20
48,DTLZ1,DTLZ1_m3_e250_s159,MOEAD,3,250,159,dvl_framework,0.30810100,0.21376689,1.44180419,1.79203972,0.03344036,0.00966131,26
24,DTLZ1,DTLZ1_m10_e500_s112,MOEAD,10,500,112,dvl_framework,0.95881069,0.04545901,1.78189755,0.78052115,0.15483381,0.00999482,20
54,DTLZ1,DTLZ1_m3_e500_s227,MOEAD,3,500,227,dvl_framework,0.22810525,0.26248422,17.58368336,22.83562558,0.06331833,0.00186453,24
6,DTLZ1,DTLZ1_m10_e1000_s200,MOEAD,10,1000,200,dvl_framework,0.97617752,0.06793507,1.74767470,1.41848928,0.21822372,0.00210128,20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
220,DTLZ4,DTLZ4_m3_e1000_s410,NSGAIII,3,1000,410,dvl_framework,0.86701391,0.01278944,0.37603679,0.04359840,0.29542184,0.00966531,20
196,DTLZ4,DTLZ4_m10_e1500_s300,NSGAIII,10,1500,300,dvl_framework,0.99979600,0.00005072,0.50529459,0.01137378,2.23424674,0.06467815,20
226,DTLZ4,DTLZ4_m3_e1500_s410,NSGAIII,3,1500,410,dvl_framework,0.87798482,0.01089295,0.33025737,0.03624030,0.55162066,0.02049588,20
184,DTLZ4,DTLZ4_m10_e10000_s300,NSGAIII,10,10000,300,dvl_framework,0.99991842,0.00003563,0.43701583,0.00632087,27.17004812,3.37083890,19



Resumo Fora do Framework:


,problem,label,algorithm,m,max_evaluations,sample_size,mode,hv_mean,hv_std,igd_mean,igd_std,cpu_time_mean,cpu_time_std,successful_runs
19,DTLZ1,DTLZ1_m10_e250_s50,MOEAD,10,250,50,pure_moea,0.85001293,0.10575858,1.83109549,0.94219898,0.09096148,0.00997099,20
49,DTLZ1,DTLZ1_m3_e250_s159,MOEAD,3,250,159,pure_moea,0.00000000,0.00000000,183.94774611,26.79214255,0.04041680,0.00505147,26
25,DTLZ1,DTLZ1_m10_e500_s112,MOEAD,10,500,112,pure_moea,0.79073452,0.28990188,3.39073860,2.62285311,0.13275778,0.01018301,20
55,DTLZ1,DTLZ1_m3_e500_s227,MOEAD,3,500,227,pure_moea,0.00000000,0.00000000,151.17153521,17.57655193,0.07795971,0.00429506,24
7,DTLZ1,DTLZ1_m10_e1000_s200,MOEAD,10,1000,200,pure_moea,0.93048316,0.16303793,2.16522635,2.10705866,0.20940604,0.01001652,20
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221,DTLZ4,DTLZ4_m3_e1000_s410,NSGAIII,3,1000,410,pure_moea,0.80620268,0.01902080,0.44846763,0.04688560,0.47392221,0.02368900,20
197,DTLZ4,DTLZ4_m10_e1500_s300,NSGAIII,10,1500,300,pure_moea,0.99979300,0.00008832,0.50292875,0.01040211,2.58413706,0.10815122,20
227,DTLZ4,DTLZ4_m3_e1500_s410,NSGAIII,3,1500,410,pure_moea,0.82515542,0.01507370,0.38488912,0.04136558,0.72928936,0.02005998,20
185,DTLZ4,DTLZ4_m10_e10000_s300,NSGAIII,10,10000,300,pure_moea,0.99995000,0.00002333,0.43194397,0.00558357,27.17331170,3.40719052,19
